# GeoSlide — Phase 5: Model Training

**Project:** GeoSlide — Wireless Sensor Network (WSN) Landslide Prediction
**Phase:** 5 of N — Model Training
**Dataset:** `wsn_landslide_data.csv`

## Scope of this Notebook

This notebook is scoped **strictly to model training**. It performs the following steps only:

1. Load the prepared WSN landslide dataset
2. Separate features and target label
3. Split the data into training and test sets (80/20, `random_state=42`)
4. Scale features where applicable using `StandardScaler`
5. Train five classification models:
   - K-Nearest Neighbors (KNN)
   - Gaussian Naive Bayes
   - Support Vector Machine (RBF kernel, `probability=True`)
   - Random Forest
   - Artificial Neural Network (Dense 64 → Dense 32 → Dense 1)
6. Generate and save predictions for every trained model
7. Persist all trained models to the `models/` directory (`joblib` for scikit-learn models, `model.save()` for the ANN)

**Explicitly out of scope for this notebook** (reserved for a later evaluation phase):
- Model comparison
- Confusion matrices
- ROC curves
- Precision / recall / F1-score reporting
- SHAP explainability
- Hyperparameter tuning
- Evaluation plots

Only minimal, basic verification (e.g. shape checks, a quick accuracy sanity check) is used where needed for debugging — not for model comparison or selection.


## 1. Environment Setup

Install and import the libraries required for data handling, preprocessing, the classical ML models, the neural network, and model persistence.

If running this notebook in **Google Colab**, the `pip install` cell below ensures all dependencies are available.


In [ ]:
# Uncomment the line below if running in a fresh Google Colab environment
# !pip install -q scikit-learn tensorflow joblib pandas numpy

import os
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0


## 2. Output Directory Setup

Create the `models/` directory (for saved model artifacts) and a `predictions/` directory (for saved prediction outputs), if they do not already exist.


In [ ]:
MODELS_DIR = "models"
PREDICTIONS_DIR = "predictions"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

print(f"Models will be saved to: ./{MODELS_DIR}/")
print(f"Predictions will be saved to: ./{PREDICTIONS_DIR}/")


Models will be saved to: ./models/
Predictions will be saved to: ./predictions/


## 3. Load Dataset

Load `wsn_landslide_data.csv` into a pandas DataFrame and perform a quick structural check (shape, dtypes, missing values). This is a basic sanity check only — no exploratory data analysis or feature engineering is performed here, as that belongs to an earlier phase of the project.


In [ ]:
DATA_PATH = "wsn_landslide_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum().sum(), "total missing values")

df.head()


Dataset shape: (9864, 35)

Missing values per column:
0 total missing values


,Rainfall_mm,Slope_Angle,Soil_Saturation,Vegetation_Cover,Rainfall_3Day,Rainfall_7Day,Aspect,Elevation_m,NDVI_Index,Land_Use_Urban,...,Soil_Type_Silt,Soil_Type_Clay,Pore_Water_Pressure_kPa,Soil_Moisture_Content,Microseismic_Activity,Acoustic_Emission_dB,Soil_Strain,Soil_Temperature_C,TDR_Reflection_Index,Label
0,164.695364,59.783332,0.821479,0.107339,260.138381,79.297169,346.674199,733.776448,0.191948,1,...,1,0,133.943194,0.143732,0.290945,51.021834,0.005167,22.760036,0.799847,1
1,34.908086,15.153084,0.100428,0.960150,510.295547,247.923576,104.462371,467.708643,0.798321,1,...,0,1,90.788608,0.266484,0.651758,39.837282,0.003443,15.558373,1.181071,0
2,38.761727,13.135384,0.286526,0.833608,297.730266,194.327012,336.671287,1880.826807,0.479456,1,...,0,1,83.041150,0.129426,0.440714,68.902366,0.009999,6.205760,1.184971,0
3,32.199977,10.674734,0.255230,0.847569,231.640610,295.139546,300.742864,964.080336,-0.084314,1,...,1,0,196.089305,0.240198,0.794001,80.196960,0.003850,25.486545,0.677944,0
4,218.114032,48.839269,0.720071,0.018383,330.278249,301.288824,155.550502,165.699102,0.810869,0,...,1,1,106.778890,0.345724,0.009160,99.919786,0.003061,7.270319,0.882642,1


## 4. Feature and Target Separation

The target column is `Label` (binary: `1` = landslide event, `0` = no landslide event). All remaining columns are treated as model features.


In [ ]:
TARGET_COL = "Label"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("\nTarget class distribution:")
print(y.value_counts())


Feature matrix shape: (9864, 34)
Target vector shape: (9864,)

Target class distribution:
Label
1    4959
0    4905
Name: count, dtype: int64


## 5. Train / Test Split

An 80/20 train-test split is used, with `random_state=42` for reproducibility. `stratify=y` is used to preserve the class balance of the target variable across both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


X_train shape: (7891, 34)
X_test shape: (1973, 34)
y_train shape: (7891,)
y_test shape: (1973,)


## 6. Feature Scaling

`StandardScaler` is fit **only on the training data** and then used to transform both the training and test sets, preventing data leakage.

Scaled features (`X_train_scaled` / `X_test_scaled`) are used for models that are sensitive to feature magnitude:
- K-Nearest Neighbors
- Support Vector Machine
- Artificial Neural Network

Unscaled features (`X_train` / `X_test`) are used for models that are not sensitive to feature scale:
- Gaussian Naive Bayes
- Random Forest

The fitted scaler is saved to `models/` so that the same transformation can be reapplied consistently during a later evaluation or deployment phase.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaler_path = os.path.join(MODELS_DIR, "scaler.joblib")
joblib.dump(scaler, scaler_path)

print(f"StandardScaler fitted and saved to: {scaler_path}")


StandardScaler fitted and saved to: models/scaler.joblib


## 7. Model 1 — K-Nearest Neighbors (KNN)

KNN is trained on the **scaled** feature set, since it relies on distance calculations that are sensitive to feature magnitude. Default hyperparameters are used — no tuning is performed in this phase.


In [ ]:
knn_model = KNeighborsClassifier()
knn_model.fit(X_train_scaled, y_train)

knn_predictions = knn_model.predict(X_test_scaled)
knn_pred_proba = knn_model.predict_proba(X_test_scaled)

# Basic debugging check only (not a comparison metric)
print("KNN training complete.")
print("Predictions generated for", len(knn_predictions), "test samples.")


KNN training complete.
Predictions generated for 1973 test samples.


## 8. Model 2 — Gaussian Naive Bayes

Gaussian Naive Bayes is trained on the **unscaled** feature set. It assumes features are normally distributed and does not require standardization to function correctly.


In [ ]:
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

nb_predictions = nb_model.predict(X_test)
nb_pred_proba = nb_model.predict_proba(X_test)

print("Gaussian Naive Bayes training complete.")
print("Predictions generated for", len(nb_predictions), "test samples.")


Gaussian Naive Bayes training complete.
Predictions generated for 1973 test samples.


## 9. Model 3 — Support Vector Machine (RBF Kernel)

The SVM uses an RBF kernel with `probability=True`, enabling probability estimates for use in later evaluation phases. It is trained on the **scaled** feature set, as SVMs are sensitive to feature magnitude.


In [ ]:
svm_model = SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
svm_model.fit(X_train_scaled, y_train)

svm_predictions = svm_model.predict(X_test_scaled)
svm_pred_proba = svm_model.predict_proba(X_test_scaled)

print("SVM (RBF) training complete.")
print("Predictions generated for", len(svm_predictions), "test samples.")


SVM (RBF) training complete.
Predictions generated for 1973 test samples.


## 10. Model 4 — Random Forest

Random Forest is trained on the **unscaled** feature set, as tree-based ensemble methods are invariant to feature scaling. `random_state=42` is set for reproducibility.


In [ ]:
rf_model = RandomForestClassifier(random_state=RANDOM_STATE)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)

print("Random Forest training complete.")
print("Predictions generated for", len(rf_predictions), "test samples.")


Random Forest training complete.
Predictions generated for 1973 test samples.


## 11. Model 5 — Artificial Neural Network

A feed-forward neural network is built with the architecture:

```
Input → Dense(64, ReLU) → Dense(32, ReLU) → Dense(1, Sigmoid)
```

The network is trained on the **scaled** feature set for binary classification, using binary cross-entropy loss and the Adam optimizer. No hyperparameter tuning is performed — a single fixed configuration is used.


In [ ]:
n_features = X_train_scaled.shape[1]

ann_model = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(64, activation="relu", name="Dense64"),
    layers.Dense(32, activation="relu", name="Dense32"),
    layers.Dense(1, activation="sigmoid", name="Dense1_Output"),
])

ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Dense64 (Dense)                 │ (None, 64)             │         2,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense32 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dense1_Output (Dense)           │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,353 (17.00 KB)

 Trainable params: 4,353 (17.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = ann_model.fit(
    X_train_scaled, y_train,
    validation_split=0.1,
    epochs=50,
    batch_size=32,
    verbose=1
)


Epoch 1/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9492 - loss: 0.2115 - val_accuracy: 0.9696 - val_loss: 0.1503
Epoch 2/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9696 - loss: 0.1422 - val_accuracy: 0.9696 - val_loss: 0.1496
Epoch 3/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9696 - loss: 0.1360 - val_accuracy: 0.9696 - val_loss: 0.1505
Epoch 4/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9696 - loss: 0.1313 - val_accuracy: 0.9696 - val_loss: 0.1517
Epoch 5/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9696 - loss: 0.1269 - val_accuracy: 0.9696 - val_loss: 0.1531
Epoch 6/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9696 - loss: 0.1228 - val_accuracy: 0.9696 - val_loss: 0.1546
Epoch 7/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9696 - loss: 0.1185 - val_accuracy: 0.9696 - val_loss: 0.1566
Epoch 8/50
222/222 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9696 - loss: 0.1140 - val_accuracy: 0.

In [ ]:
ann_pred_proba = ann_model.predict(X_test_scaled).ravel()
ann_predictions = (ann_pred_proba >= 0.5).astype(int)

print("ANN training complete.")
print("Predictions generated for", len(ann_predictions), "test samples.")


## 12. Save Predictions for All Models

Predictions (class labels and, where available, positive-class probabilities) from every trained model are saved to a single CSV file for use in the subsequent evaluation phase. This step performs no scoring, comparison, or ranking of the models.


In [ ]:
predictions_df = pd.DataFrame({
    "y_true": y_test.reset_index(drop=True),
    "knn_pred": knn_predictions,
    "knn_proba": knn_pred_proba[:, 1],
    "nb_pred": nb_predictions,
    "nb_proba": nb_pred_proba[:, 1],
    "svm_pred": svm_predictions,
    "svm_proba": svm_pred_proba[:, 1],
    "rf_pred": rf_predictions,
    "rf_proba": rf_pred_proba[:, 1],
    "ann_pred": ann_predictions,
    "ann_proba": ann_pred_proba,
})

predictions_path = os.path.join(PREDICTIONS_DIR, "phase5_model_predictions.csv")
predictions_df.to_csv(predictions_path, index=False)

print(f"Predictions for all 5 models saved to: {predictions_path}")
predictions_df.head()


## 13. Save Trained Models

- The four scikit-learn models (KNN, Naive Bayes, SVM, Random Forest) are saved using **`joblib`**.
- The ANN is saved using **`model.save()`** (Keras native/TensorFlow `SavedModel` format).
- The fitted `StandardScaler` was already saved in Section 6, and is required to correctly preprocess new data for the scaled models (KNN, SVM, ANN) in the evaluation phase.

All artifacts are written to the `models/` directory.


In [ ]:
# Scikit-learn models -> joblib
joblib.dump(knn_model, os.path.join(MODELS_DIR, "knn_model.joblib"))
joblib.dump(nb_model, os.path.join(MODELS_DIR, "naive_bayes_model.joblib"))
joblib.dump(svm_model, os.path.join(MODELS_DIR, "svm_model.joblib"))
joblib.dump(rf_model, os.path.join(MODELS_DIR, "random_forest_model.joblib"))

# ANN -> TensorFlow / Keras native save format
ann_model.save(os.path.join(MODELS_DIR, "ann_model.keras"))

print("All models saved successfully to the 'models/' directory:")
for f in sorted(os.listdir(MODELS_DIR)):
    print(" -", f)


## 14. Summary — Phase 5 Complete

### Models Trained
| # | Model | Feature Set Used |
|---|-------|-------------------|
| 1 | K-Nearest Neighbors | Scaled |
| 2 | Gaussian Naive Bayes | Unscaled |
| 3 | Support Vector Machine (RBF, `probability=True`) | Scaled |
| 4 | Random Forest | Unscaled |
| 5 | Artificial Neural Network (Dense64 → Dense32 → Dense1) | Scaled |

### Files Saved

**`models/`**
- `scaler.joblib` — fitted `StandardScaler`
- `knn_model.joblib`
- `naive_bayes_model.joblib`
- `svm_model.joblib`
- `random_forest_model.joblib`
- `ann_model.keras` — TensorFlow/Keras native saved model

**`predictions/`**
- `phase5_model_predictions.csv` — true labels plus predicted labels and probabilities for all five models

### Ready for Evaluation

All five models have been trained, their predictions captured, and all artifacts persisted to disk. This notebook intentionally stops here: no model comparison, confusion matrices, ROC curves, precision/recall/F1 reporting, SHAP explainability, hyperparameter tuning, or evaluation plots have been performed.

The saved models, scaler, and predictions file in this notebook are now ready to be consumed by a dedicated **Phase 6: Model Evaluation** notebook.
